In [104]:
from pathlib import Path
import cv2
import pandas as pd
from ultralytics import YOLO

In [105]:
Path("../data/test.mp4").exists()

True

In [ ]:

# -------- CONFIG --------
VIDEO_PATH = "../data/test.mp4"         
OUT_VIDEO = "../data/tracked_chickens.mp4"
OUT_TABLE = "../data/chicken_tracks.parquet"

CONF_THRESH = 0.25                      # detection confidence threshold
IOU_THRESH = 0.45                       # NMS IoU threshold
TRACKER = "bytetrack.yaml"              # built-in tracker config
DEVICE = "mps"                           # set to 0 for GPU if available, else None for CPU


# Filter detections to these classes (set to None to keep all)
ALLOWED_CLASSES = {"bird"}              # YOLOv8 COCO label; chickens are usually "bird"

# -------- STABLE LABELS MANAGER --------
STABLE_NAMES = ["Chicken A", "Chicken B", "Chicken C"]

# Per-name state
name_state = {n: {"track_id": None, "last_pos": None, "last_seen": -1} for n in STABLE_NAMES}

# Map current tracker IDs -> friendly names
id2name = {}

# Tunable heuristics (adjust to your resolution and motion)
ASSIGN_DIST_THRESH = 400   # pixels; max distance to consider "same chicken"
MISS_TOLERANCE = 60        # frames; how long we keep a slot alive without observations

In [107]:

# -------- UTILITIES --------
def draw_label(img, label, x1, y1):
    """Draw a filled rectangle behind text for readability, positioned above the box."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.6
    thickness = 1
    (w, h), _ = cv2.getTextSize(label, font, scale, thickness)
    top_left = (int(x1), int(y1) - h - 6)
    bottom_right = (int(x1) + w + 6, int(y1))
    cv2.rectangle(img, top_left, bottom_right, (0, 0, 0), -1)
    cv2.putText(img, label, (int(x1) + 3, int(y1) - 4), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)


def assign_stable_name(track_id, cx, cy, frame_idx, used_names_this_frame):
    """Return a stable name for this detection and update state.
       If track_id changes, rebind to the nearest existing chicken name.
    """
    # 1) If we already know this track_id, reuse its friendly name.
    if track_id in id2name:
        name = id2name[track_id]
        name_state[name]["last_pos"] = (cx, cy)
        name_state[name]["last_seen"] = frame_idx
        name_state[name]["track_id"] = track_id
        used_names_this_frame.add(name)
        return name

    # 2) Try to match to nearest active name by position (not too old, not already used).
    nearest_name, nearest_dist = None, float("inf")
    for name, st in name_state.items():
        if st["last_pos"] is None:
            continue
        if frame_idx - st["last_seen"] > MISS_TOLERANCE:
            continue
        if name in used_names_this_frame:
            continue
        px, py = st["last_pos"]
        d = ((cx - px)**2 + (cy - py)**2) ** 0.5
        if d < nearest_dist:
            nearest_name, nearest_dist = name, d

    if nearest_name is not None and nearest_dist <= ASSIGN_DIST_THRESH:
        # Rebind the friendly name from old tracker ID to the new track_id.
        old_id = name_state[nearest_name]["track_id"]
        if old_id is not None and old_id in id2name:
            del id2name[old_id]
        if track_id != -1:  # don't persist -1 in the map
            id2name[track_id] = nearest_name
        name_state[nearest_name]["track_id"] = track_id if track_id != -1 else name_state[nearest_name]["track_id"]
        name_state[nearest_name]["last_pos"] = (cx, cy)
        name_state[nearest_name]["last_seen"] = frame_idx
        used_names_this_frame.add(nearest_name)
        return nearest_name

    # 3) Otherwise, grab an unused/stale slot.
    for name, st in name_state.items():
        if name in used_names_this_frame:
            continue
        if st["last_pos"] is None or frame_idx - st["last_seen"] > MISS_TOLERANCE:
            if track_id != -1:
                id2name[track_id] = name
                name_state[name]["track_id"] = track_id
            name_state[name]["last_pos"] = (cx, cy)
            name_state[name]["last_seen"] = frame_idx
            used_names_this_frame.add(name)
            return name

    # 4) Fallback: raw ID (rare; e.g., >3 detections or unusual frame)
    return f"Chicken {track_id}"

In [108]:

# -------- LOAD MODEL --------
model = YOLO("yolov8s.pt")  # small & fast; try 'yolov8s.pt' for better accuracy


In [109]:

# Get video metadata for writer
cap_meta = cv2.VideoCapture(VIDEO_PATH)
fps = cap_meta.get(cv2.CAP_PROP_FPS)
fps = float(fps) if fps and fps > 0 else 30.0
width = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_meta.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (width, height))

records = []  # will store per-detection per-frame rows
frame_idx = -1


In [110]:
# -------- TRACKING LOOP --------
# The generator yields a Result per frame with tracked boxes (boxes.id).
for result in model.track(
    source=VIDEO_PATH,
    stream=True,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    tracker=TRACKER,
    persist=True,             # keep tracker state across frames
    device=DEVICE
):
    
    used_names_this_frame = set()

    frame_idx += 1
    frame = result.orig_img.copy()  # BGR image

    if result.boxes is None or len(result.boxes) == 0:
        out.write(frame)
        continue

    boxes = result.boxes

    # Iterate detections for this frame

    for b in boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        w_box = x2 - x1
        h_box = y2 - y1

        conf = float(b.conf[0]) if b.conf is not None else 0.0
        cls_id = int(b.cls[0]) if b.cls is not None else -1
        cls_name = model.names.get(cls_id, "unknown")

        if ALLOWED_CLASSES and cls_name not in ALLOWED_CLASSES:
            continue

        track_id = int(b.id[0]) if hasattr(b, "id") and b.id is not None else -1

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        cx_norm = cx / width
        cy_norm = cy / height

        # --- Assign stable friendly name ---
        pretty_name = assign_stable_name(track_id, cx, cy, frame_idx, used_names_this_frame)

        # Draw
        color = (0, 255, 0)
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"{pretty_name} conf={conf:.2f} id={track_id}"
        draw_label(frame, label, x1, y1)

        # Save row
        records.append({
            "frame": frame_idx,
            "track_id": track_id,
            "chicken_name": pretty_name,
            "class_id": cls_id,
            "class_name": cls_name,
            "confidence": conf,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "width": w_box, "height": h_box,
            "cx": cx, "cy": cy,
            "cx_norm": cx_norm, "cy_norm": cy_norm
        })


    # Write the augmented frame
    out.write(frame)



video 1/1 (frame 1/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 1 cat, 19.5ms
video 1/1 (frame 2/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 1 cat, 7.4ms
video 1/1 (frame 3/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 1 cat, 8.2ms
video 1/1 (frame 4/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 1 cat, 8.7ms
video 1/1 (frame 5/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 1 cat, 6.1ms
video 1/1 (frame 6/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 (no detections), 6.1ms
video 1/1 (frame 7/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 (no detections), 7.2ms
video 1/1 (frame 8/733) /Users/prince/proj/realtime-detection-v2/notebook/../data/test.mp4: 544x640 (no detections), 6.5ms
video 1/1 (frame 9/733) /Users/prince/proj/realtime-detection-v2

In [111]:

# -------- SAVE OUTPUTS --------
out.release()

df = pd.DataFrame.from_records(records)
# Helpful indexing for analysis:
df.sort_values(["track_id", "frame"], inplace=True)
df.to_parquet(OUT_TABLE, index=False)  # requires pyarrow

print(f"Saved video to: {OUT_VIDEO}")
print(f"Saved tracks to: {OUT_TABLE}")
print(df.head())


Saved video to: ../data/tracked_chickens.mp4
Saved tracks to: ../data/chicken_tracks.parquet
     frame  track_id chicken_name  class_id class_name  confidence          x1          y1          x2          y2       width      height          cx          cy   cx_norm   cy_norm
25      93        -1    Chicken C        14       bird    0.395981  388.738800  337.324188  579.747498  531.038757  191.008698  193.714569  484.243149  434.181473  0.687845  0.753787
26      93        -1    Chicken A        14       bird    0.298754  390.033722  453.663116  521.634216  529.485474  131.600494   75.822357  455.833969  491.574295  0.647491  0.853428
82     118        -1    Chicken B        14       bird    0.537437  287.114624  194.760117  439.108124  257.200500  151.993500   62.440384  363.111374  225.980309  0.515783  0.392327
84     128        -1    Chicken C        14       bird    0.273732  510.502045  416.613373  617.637756  474.344025  107.135712   57.730652  564.069901  445.478699  0.801236  0

In [112]:
df

,frame,track_id,chicken_name,class_id,class_name,confidence,x1,y1,x2,y2,width,height,cx,cy,cx_norm,cy_norm
25,93,-1,Chicken C,14,bird,0.395981,388.738800,337.324188,579.747498,531.038757,191.008698,193.714569,484.243149,434.181473,0.687845,0.753787
26,93,-1,Chicken A,14,bird,0.298754,390.033722,453.663116,521.634216,529.485474,131.600494,75.822357,455.833969,491.574295,0.647491,0.853428
82,118,-1,Chicken B,14,bird,0.537437,287.114624,194.760117,439.108124,257.200500,151.993500,62.440384,363.111374,225.980309,0.515783,0.392327
84,128,-1,Chicken C,14,bird,0.273732,510.502045,416.613373,617.637756,474.344025,107.135712,57.730652,564.069901,445.478699,0.801236,0.773401
817,560,-1,Chicken B,14,bird,0.295694,97.788864,208.350189,168.488174,319.089752,70.699310,110.739563,133.138519,263.719971,0.189117,0.457847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,729,266,Chicken B,14,bird,0.732389,349.669891,173.726578,511.051941,266.267456,161.382050,92.540878,430.360916,219.997017,0.611308,0.381939
998,730,266,Chicken B,14,bird,0.625177,354.671631,173.171463,508.285278,264.206726,153.613647,91.035263,431.478455,218.689095,0.612896,0.379669
1000,731,266,Chicken B,14,bird,0.452092,359.382782,174.069138,506.305359,263.936554,146.922577,89.867416,432.844070,219.002846,0.614835,0.380213
1002,732,266,Chicken B,14,bird,0.312556,361.249207,174.791428,506.289429,266.502441,145.040222,91.711014,433.769318,220.646935,0.616150,0.383068


In [113]:
df.chicken_name.unique()

array(['Chicken C', 'Chicken A', 'Chicken B', 'Chicken 57', 'Chicken 62'], dtype=object)